In [ ]:
#CELULA 1: Imports, Variaveis de Ambiente, GCP Client e Configuracoes Globais

import os
import io
import zipfile
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client GCP
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais Dinâmicos com Data Piso
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)

DATA_PISO_BACKFILL = datetime.strptime("2026-01-01", "%Y-%m-%d").date()

bucket_bronze = client.bucket(BUCKET_BRONZE)
bucket_silver = client.bucket(BUCKET_SILVER)

pd.set_option("display.max_columns", None)

print(f"✔ Client GCP inicializado para CAMADA SILVER!")
print(f"✔ Source Bronze: gs://{BUCKET_BRONZE}")
print(f"✔ Target Silver: gs://{BUCKET_SILVER}")
print(f"✔ Data Limite D-1: {D1_LIMITE.strftime('%d/%m/%Y')}")

In [ ]:
#CELULA 2: Pipeline Silver - Leitura da Bronze, Filtros de Negocio e Persistencia em Parquet

def salvar_dataframe_parquet_gcs(df: pd.DataFrame, gcs_path: str):
    """Converte o DataFrame para Parquet em memória e faz o stream para a Silver."""
    table = pa.Table.from_pandas(df)
    buffer = io.BytesIO()
    pq.write_table(table, buffer, compression='snappy')
    buffer.seek(0)
    
    blob = bucket_silver.blob(gcs_path)
    blob.upload_from_string(buffer.getvalue(), content_type="application/octet-stream")
    print(f" Success: gs://{BUCKET_SILVER}/{gcs_path}")

def processar_silver_fundos_elegiveis_mes(ano_mes: str) -> bool:
    """
    Lê o ZIP do CDA_FI direto da Bronze, aplica os 5 filtros e grava em Parquet na Silver.
    """
    ano_str = ano_mes[:4]
    mes_str = ano_mes[4:6]
    
    path_bronze_zip = f"cvm/cda_fi/ano={ano_str}/mes={mes_str}/cda_fi_{ano_mes}.zip"
    blob_bronze = bucket_bronze.blob(path_bronze_zip)
    
    if not blob_bronze.exists():
        print(f"⚠ Arquivo da Bronze {path_bronze_zip} ainda não existe no GCS. Pulando...")
        return False

    print(f"🔄 Processando Silver [{ano_mes}]: Lendo cda_fi_{ano_mes}.zip da Bronze...")
    content_zip = blob_bronze.download_as_bytes()
    
    with zipfile.ZipFile(io.BytesIO(content_zip)) as z:
        # 1. Patrimônio Líquido
        df_pl = pd.read_csv(z.open(f'cda_fi_PL_{ano_mes}.csv'), sep=";", encoding="ISO-8859-1")
        df_pl = df_pl[['CNPJ_FUNDO_CLASSE', 'DENOM_SOCIAL', 'VL_PATRIM_LIQ']].drop_duplicates()
        df_pl = df_pl[df_pl['VL_PATRIM_LIQ'] > 0]

        # 2. BLC_2: FIDCs
        df_blc2 = pd.read_csv(z.open(f'cda_fi_BLC_2_{ano_mes}.csv'), sep=";", encoding="ISO-8859-1")
        fidc_sum = df_blc2[df_blc2['TP_ATIVO'].str.contains('FIDC', na=False, case=False)] \
                    .groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_FIDC'})

        # 3. BLC_4: Ações
        df_blc4 = pd.read_csv(z.open(f'cda_fi_BLC_4_{ano_mes}.csv'), sep=";", encoding="ISO-8859-1")
        acoes_sum = df_blc4[df_blc4['TP_ATIVO'].str.contains('AÇÕES|ACOES|UNIT', na=False, case=False)] \
                     .groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_ACOES'})

        # 4. BLC_5 + BLC_6: Crédito Privado
        df_blc5 = pd.read_csv(z.open(f'cda_fi_BLC_5_{ano_mes}.csv'), sep=";", encoding="ISO-8859-1")
        df_blc6 = pd.read_csv(z.open(f'cda_fi_BLC_6_{ano_mes}.csv'), sep=";", encoding="ISO-8859-1")
        credito_sum = pd.concat([df_blc5, df_blc6]).groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_CREDITO'})

    # Consolidando matriz
    df_mestre = df_pl.merge(fidc_sum, on='CNPJ_FUNDO_CLASSE', how='left') \
                     .merge(acoes_sum, on='CNPJ_FUNDO_CLASSE', how='left') \
                     .merge(credito_sum, on='CNPJ_FUNDO_CLASSE', how='left').fillna(0)

    # Cálculo dos Percentuais
    df_mestre['PCT_CREDITO'] = (df_mestre['VL_CREDITO'] / df_mestre['VL_PATRIM_LIQ']).astype('float64')
    df_mestre['PCT_FIDC'] = (df_mestre['VL_FIDC'] / df_mestre['VL_PATRIM_LIQ']).astype('float64')
    df_mestre['PCT_ACOES'] = (df_mestre['VL_ACOES'] / df_mestre['VL_PATRIM_LIQ']).astype('float64')

    # Aplicação dos 5 Filtros de Crédito Elegíveis
    cond_credito = df_mestre['PCT_CREDITO'] >= 0.10
    cond_nome = ~df_mestre['DENOM_SOCIAL'].str.contains('INVESTIMENTO NO EXTERIOR| IE | EXTERIOR', case=False, regex=True, na=False)
    cond_fidc = df_mestre['PCT_FIDC'] <= 0.30
    cond_acoes = df_mestre['PCT_ACOES'] <= 0.03

    df_elegiveis = df_mestre[cond_credito & cond_nome & cond_fidc & cond_acoes].copy()
    df_elegiveis['ANO_MES_REFERENCIA'] = ano_mes
    df_elegiveis['DT_PROCESSAMENTO'] = datetime.now().strftime("%Y-%m-%d")

    # Caminho do Parquet na Camada Silver
    silver_path = f"cvm/fundos_credito_elegiveis/ano={ano_str}/mes={mes_str}/fundos_elegiveis_{ano_mes}.parquet"
    salvar_dataframe_parquet_gcs(df_elegiveis, silver_path)
    return True

def executar_pipeline_silver_elegiveis():
    print(f"\n=== EXECUTANDO TRANSFORMACAO SILVER (FUNDOS ELEGIVEIS) ===")
    
    ano_inicio = DATA_PISO_BACKFILL.year
    ano_fim = D1_LIMITE.year
    
    for ano in range(ano_inicio, ano_fim + 1):
        # Para cada ano da janela de piso, varre os meses até o limite D-1
        mes_limite = D1_LIMITE.month if ano == ano_fim else 12
        
        for mes in range(1, mes_limite + 1):
            ano_mes = f"{ano}{mes:02d}"
            silver_path = f"cvm/fundos_credito_elegiveis/ano={ano}/mes={mes:02d}/fundos_elegiveis_{ano_mes}.parquet"
            blob_silver = bucket_silver.blob(silver_path)
            
            # Checagem de Idempotência:
            # Se já existir o Parquet na Silver e for mês passado -> Pula
            if blob_silver.exists() and (ano < ano_fim or mes < mes_limite):
                print(f"✔ [SILVER] Parquet do mês {ano_mes} já consolidado na Silver. Pulando...")
                continue
                
            processar_silver_fundos_elegiveis_mes(ano_mes)

# Executa o pipeline para gerar os Parquets da Silver
executar_pipeline_silver_elegiveis()

In [ ]:
# CÉLULA EXPLORATÓRIA EM ETAPAS: Filtros Sequenciais mantendo Masters e FICs

import pandas as pd

print("=== INICIANDO PIPELINE DE FILTRAGEM PASSO-A-PASSO (MASTERS + FICS) ===")

# ==============================================================================
# ETAPA 0: BASE BRUTA COMPLETA (Fundos com PL > 0)
# ==============================================================================
df_etapa_0_base = df_screening.copy()
qtd_etapa_0 = len(df_etapa_0_base)
print(f"📊 [Etapa 0] Base Bruta Total (PL > 0): {qtd_etapa_0:,} fundos")


# ==============================================================================
# ETAPA 1: FILTRO DE CRÉDITO PRIVADO (DIRETO + INDIRETO VIA FICs)
# - Masters: >= 10% do PL em papéis diretos (Debêntures, LF, CDB, CRI, CRA, etc.)
# - FICs: >= 50% do PL alocado no BLC_7 (Cotas de Fundos)
# ==============================================================================
cond_e1_credito_direto = df_etapa_0_base['PCT_CREDITO'] >= 0.10
cond_e1_fic_cotas = df_etapa_0_base.get('PCT_COTAS_BLC7', 0) >= 0.50

cond_e1_credito_combinado = cond_e1_credito_direto | cond_e1_fic_cotas
df_etapa_1_credito = df_etapa_0_base[cond_e1_credito_combinado].copy()

qtd_etapa_1 = len(df_etapa_1_credito)
removidos_e1 = qtd_etapa_0 - qtd_etapa_1
print(f"🔹 [Etapa 1] Mínimo 10% Crédito (Direto + FICs Mantidos): {qtd_etapa_1:,} fundos (Filtrados/Removidos: {removidos_e1:,})")


# ==============================================================================
# ETAPA 2: FILTRO DE INVESTIMENTO NO EXTERIOR (Exclui Fundos IE / Exterior)
# ==============================================================================
cond_e2_exterior = ~df_etapa_1_credito['DENOM_SOCIAL'].str.contains('EXTERIOR|IE', case=False, na=False) & \
                    (df_etapa_1_credito.get('INVEST_EXTER', 'N') != 'S')

df_etapa_2_exterior = df_etapa_1_credito[cond_e2_exterior].copy()

qtd_etapa_2 = len(df_etapa_2_exterior)
removidos_e2 = qtd_etapa_1 - qtd_etapa_2
print(f"🔹 [Etapa 2] Exclusão de Investimento no Exterior: {qtd_etapa_2:,} fundos (Filtrados/Removidos: {removidos_e2:,})")


# ==============================================================================
# ETAPA 3: FILTRO DE LIMITE EM FIDC (<= 30% do PL)
# Nota: FICs puros que possuem 0% de FIDC direto passam normalmente
# ==============================================================================
cond_e3_fidc = df_etapa_2_exterior.get('PCT_FIDC', 0) <= 0.30
df_etapa_3_fidc = df_etapa_2_exterior[cond_e3_fidc].copy()

qtd_etapa_3 = len(df_etapa_3_fidc)
removidos_e3 = qtd_etapa_2 - qtd_etapa_3
print(f"🔹 [Etapa 3] Limite de até 30% em FIDC: {qtd_etapa_3:,} fundos (Filtrados/Removidos: {removidos_e3:,})")


# ==============================================================================
# ETAPA 4: FILTRO DE EXCLUSÃO DE SOBERANOS PUROS (> 95% do PL ou Nome Soberano)
# Nota: Mantém fundos de crédito que usam LFT/Tesouro como reserva de caixa/liquidez
# ==============================================================================
cond_e4_nao_soberano_puro = (df_etapa_3_fidc.get('PCT_SOBERANO', 0) < 0.95) & \
                            (~df_etapa_3_fidc['DENOM_SOCIAL'].str.contains('TESOURO SELIC|DI SIMPLES|SOBERANO', case=False, na=False))

df_etapa_4_soberano = df_etapa_3_fidc[cond_e4_nao_soberano_puro].copy()

qtd_etapa_4 = len(df_etapa_4_soberano)
removidos_e4 = qtd_etapa_3 - qtd_etapa_4
print(f"🔹 [Etapa 4] Exclusão de Fundos 100% Soberanos/DI Puros: {qtd_etapa_4:,} fundos (Filtrados/Removidos: {removidos_e4:,})")


# ==============================================================================
# ETAPA 5: FILTRO DE LIMITE EM AÇÕES (<= 3% do PL) -> BASE FINAL ELEGÍVEL
# ==============================================================================
cond_e5_acoes = df_etapa_4_soberano.get('PCT_ACOES', 0) <= 0.03
df_etapa_5_elegiveis = df_etapa_4_soberano[cond_e5_acoes].copy()

qtd_etapa_5 = len(df_etapa_5_elegiveis)
removidos_e5 = qtd_etapa_4 - qtd_etapa_5
print(f"🔹 [Etapa 5] Limite de até 3% em Ações/Units: {qtd_etapa_5:,} fundos (Filtrados/Removidos: {removidos_e5:,})")

print("\n==================================================")
print(f"🏆 ELEGÍVEIS FINAIS APÓS SCREENING COMPLETO (MASTERS + FICS): {qtd_etapa_5:,} fundos")
print("==================================================")

In [ ]:
# CÉLULA CORRIGIDA: ABERTURA DE ATIVOS DA ETAPA 1 (AMOSTRAGEM MAIS RECENTE)

import pandas as pd

print("=== CONSTRUINDO A AMOSTRAGEM COM A CARTEIRA MAIS RECENTE POR FUNDO ===")

# 1. Mapeamento dinâmico de colunas para tratar variações de nomes na tabela de ativos
def mapear_coluna(df, candidatos):
    for col in candidatos:
        if col in df.columns:
            return col
    raise KeyError(f"Nenhuma das colunas {candidatos} foi encontrada no DataFrame. Colunas disponíveis: {list(df.columns)}")

col_cnpj_ativos = mapear_coluna(df_ativos_detalhado, ['CNPJ_FUNDO_CLASSE', 'CNPJ_FUNDO'])
col_data_ativos = mapear_coluna(df_ativos_detalhado, ['DT_COMPTC', 'DT_COMPET', 'DT_INFC', 'DT_REGISTRO', 'DT_POSICAO'])

col_cnpj_screening = mapear_coluna(df_screening, ['CNPJ_FUNDO_CLASSE', 'CNPJ_FUNDO'])

print(f"✔ Coluna de CNPJ identificada: '{col_cnpj_screening}'")
print(f"✔ Coluna de Data identificada em df_ativos_detalhado: '{col_data_ativos}'")

# 2. Isola a data de competência mais recente disponível para cada fundo na base de ATIVOS
ultimas_datas_ativos = df_ativos_detalhado.groupby(col_cnpj_ativos)[col_data_ativos].max().reset_index()

df_ativos_mais_recente = df_ativos_detalhado.merge(
    ultimas_datas_ativos, 
    on=[col_cnpj_ativos, col_data_ativos], 
    how='inner'
)

print(f"\n📊 Total de posições na carteira mais recente: {len(df_ativos_mais_recente):,}")
print("📅 Distribuição das competências capturadas nos ativos (Sigilo vs D-15):")
print(df_ativos_mais_recente[col_data_ativos].value_counts())

# ==============================================================================
# ETAPA 1: FILTRO DE CRÉDITO PRIVADO NA AMOSTRAGEM MAIS RECENTE (MASTERS + FICS)
# ==============================================================================
# Considera >= 10% Crédito Direto OU >= 50% alocado em Cotas de Fundos (FICs / BLC_7)
cond_e1_credito_direto = df_screening['PCT_CREDITO'] >= 0.10
cond_e1_fic_cotas = df_screening.get('PCT_COTAS_BLC7', 0) >= 0.50

df_etapa_1_mais_recente = df_screening[cond_e1_credito_direto | cond_e1_fic_cotas].copy()

print(f"\n🔹 Fundos Aprovados na Etapa 1 (Masters + FICs): {len(df_etapa_1_mais_recente):,} fundos")

# ==============================================================================
# ABERTURA ORDENADA DOS TIPOS DE ATIVOS (TP_ATIVO) DA ETAPA 1 MAIS RECENTE
# ==============================================================================
cnpjs_e1 = df_etapa_1_mais_recente[col_cnpj_screening].unique()
df_ativos_e1_recente = df_ativos_mais_recente[df_ativos_mais_recente[col_cnpj_ativos].isin(cnpjs_e1)]

abertura_tp_ativo_recente = df_ativos_e1_recente['TP_ATIVO'].value_counts().reset_index()
abertura_tp_ativo_recente.columns = ['Tipo de Ativo (TP_ATIVO)', 'Quantidade de Posições']
abertura_tp_ativo_recente['% do Total'] = (abertura_tp_ativo_recente['Quantidade de Posições'] / abertura_tp_ativo_recente['Quantidade de Posições'].sum()) * 100

pd.set_option('display.max_rows', None)
print("\n=== DISTRIBUIÇÃO DOS ATIVOS DA ETAPA 1 (AMOSTRAGEM MAIS RECENTE - ORDENADO DESC) ===")
print(abertura_tp_ativo_recente.to_string(index=False, formatters={'% do Total': '{:.2f}%'.format}))